# Impulse — A2D2 Analysis: safety event search + event statistics + clips

Companion to `a2d2_ingestion.ipynb` and `a2d2_object_detection.ipynb`. It performs **event
search** over the silver-layer channels — fusing **bus signals** (speed, braking, steering)
with the **object-detection channels** (`detected_pedestrians` / `detected_cyclists` /
`detected_vehicles`) to find **safety-relevant scenarios** — computes **statistics within
those events**, and exports each event as an H.264 MP4 clip onto the volume.

## Prerequisite
Run `a2d2_ingestion.ipynb` (with `download_images=true`) and `a2d2_object_detection.ipynb`
first. The detection notebook adds the `detected_*` channels; without them only the
bus-signal events are evaluated (the perception-fused events are skipped automatically).

## License
A2D2 © Audi AG, **CC BY-ND 4.0** (https://creativecommons.org/licenses/by-nd/4.0/, source
https://www.a2d2.audi/). This notebook reads only your own ingested tables/images — it
redistributes nothing. Commit it with outputs cleared.

In [ ]:
%pip install pydantic>=2.0 scipy pillow imageio imageio-ffmpeg -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.text("clip_max_duration_s", "5", "Clip max duration (s)")
dbutils.widgets.text("clip_fps", "5", "Clip playback fps")
dbutils.widgets.text("clips_per_event", "5", "Clips per event name")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
CLIP_MAX_DURATION_S = float(dbutils.widgets.get("clip_max_duration_s") or "5")
CLIP_FPS = float(dbutils.widgets.get("clip_fps") or "5")
CLIPS_PER_EVENT = int(dbutils.widgets.get("clips_per_event") or "5")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

# Works from a cloned repo/Git folder (src on path) and as a DABs wheel-installed job.
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
CONTAINER_ID = 1
REPORT_NAME = "a2d2_analysis"
vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
clips_dir = f"{vol_root}/{REPORT_NAME}"   # event clips written here (folder = report name)
if not spark.catalog.tableExists(f"{pfx}_channels"):
    raise RuntimeError(f"{pfx}_channels not found. Run a2d2_ingestion.ipynb first.")
print("Reading from", f"{pfx}_*", "| clips ->", clips_dir)

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.sequence_of_events import SequenceOfEvents
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
import numpy as np
import pandas as pd
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name=REPORT_NAME, spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()

## 1. Channels — bus signals + perception channels

Select the bus channels used by the events, and the object-detection channels added by
`a2d2_object_detection.ipynb` (if present).

In [ ]:
speed = db.query.channel(channel_name="vehicle_speed")
acc_x = db.query.channel(channel_name="acceleration_x")
steer = db.query.channel(channel_name="steering_angle_calculated")
brake = db.query.channel(channel_name="brake_pressure")

# Perception channels exist only if a2d2_object_detection.ipynb has run.
chan_names = set(r["value"] for r in spark.read.table(f"{pfx}_channel_tags")
                 .where("key = 'channel_name'").select("value").distinct().collect())
have_counts = any(n.startswith("detected_") for n in chan_names)
have_distance = "pedestrian_nearest_distance_m" in chan_names and "vehicle_nearest_distance_m" in chan_names
print("perception:", "distance+counts" if have_distance else ("counts only" if have_counts else
      "NONE - run a2d2_object_detection.ipynb (with estimate_distance=true) for perception events"))
if have_counts:
    ped = db.query.channel(channel_name="detected_pedestrians")
    cyc = db.query.channel(channel_name="detected_cyclists")
    veh = db.query.channel(channel_name="detected_vehicles")
if have_distance:
    ped_dist = db.query.channel(channel_name="pedestrian_nearest_distance_m")
    cyc_dist = db.query.channel(channel_name="cyclist_nearest_distance_m")
    veh_dist = db.query.channel(channel_name="vehicle_nearest_distance_m")

## 2. Safety event search (proximity × dynamics)

Mere object *presence* isn't a safety event — relevance comes from **proximity + vehicle
dynamics**. Using the distance channels (`*_nearest_distance_m`) fused with the bus signals,
we define genuinely safety-relevant scenarios (thresholds tunable for the recording):

- **`evasive_maneuver`** — `SequenceOfEvents`: hard braking *followed by* sharp steering
  (brake-then-swerve), the classic collision-avoidance pattern (bus-only, always available).
- **`close_pedestrian_while_moving`** — a pedestrian within `PED_NEAR_M` while the vehicle is
  moving (> `MOVING_KMH`).
- **`close_vru_while_braking`** — a vulnerable road user within `VRU_NEAR_M` *during* hard
  braking (a braking reaction to a close VRU).
- **`close_vru_then_braking`** — `SequenceOfEvents`: a close VRU appears and is *followed by*
  braking (ordered/causal reaction).
- **`tailgating`** — time-headway to the lead vehicle below `TAILGATE_THW_S`
  (`thw = vehicle_nearest_distance_m / (vehicle_speed/3.6)`) while moving — a standard ADAS
  metric.

(If distance channels aren't available, it falls back to count-based VRU-while-braking; if no
perception at all, only the bus-signal maneuvers run.)

In [ ]:
EVENT_PAD_S = 2.5
PAD_US = int(EVENT_PAD_S * 1e6)   # .expand() works in the series' time unit (RAW = microseconds)

# Bus-signal events
hard_brake = ((brake > 20.0) | (acc_x < -2.0)).expand(PAD_US)
sharp_steer = ((steer > 45.0) | (steer < -45.0)).expand(PAD_US)
ev_brake = BasicEvent(name="hard_braking", expr=hard_brake, desc="brake>20 bar or accel_x<-2 m/s^2, +/-2.5s")
# Evasive maneuver: hard braking FOLLOWED BY sharp steering (brake-then-swerve) — the classic
# collision-avoidance pattern. A SequenceOfEvents, replacing the standalone sharp_steering.
ev_evasive = SequenceOfEvents(name="evasive_maneuver", expressions=[hard_brake, sharp_steer],
                              desc="hard braking then sharp (evasive) steering")
events = [ev_brake, ev_evasive]

# Safety thresholds (tunable; tuned for a slow urban drive, max ~22 km/h here)
PED_NEAR_M, MOVING_KMH = 20.0, 5.0
VRU_NEAR_M = 25.0
TAILGATE_THW_S, TAILGATE_MIN_KMH = 2.0, 10.0

if have_distance:
    # proximity x dynamics — genuinely safety-relevant (not mere presence)
    close_vru = (ped_dist < VRU_NEAR_M) | (cyc_dist < VRU_NEAR_M)         # VRU within range
    close_ped_moving = ((ped_dist < PED_NEAR_M) & (speed > MOVING_KMH)).expand(PAD_US)
    close_vru_braking = (close_vru.expand(PAD_US)) & hard_brake
    thw = veh_dist / (speed / 3.6)                                       # time-headway, seconds
    tailgating = ((thw < TAILGATE_THW_S) & (speed > TAILGATE_MIN_KMH)).expand(PAD_US)
    events += [
        BasicEvent(name="close_pedestrian_while_moving", expr=close_ped_moving,
                   desc=f"pedestrian <{PED_NEAR_M:.0f}m while moving >{MOVING_KMH:.0f} km/h"),
        BasicEvent(name="close_vru_while_braking", expr=close_vru_braking,
                   desc=f"VRU <{VRU_NEAR_M:.0f}m during hard braking"),
        BasicEvent(name="tailgating", expr=tailgating,
                   desc=f"time-headway <{TAILGATE_THW_S:.0f}s to lead vehicle while >{TAILGATE_MIN_KMH:.0f} km/h"),
        SequenceOfEvents(name="close_vru_then_braking", expressions=[close_vru.expand(PAD_US), hard_brake],
                   desc="close VRU, then a braking reaction (ordered)"),
    ]
elif have_counts:
    # fallback (no distance): count-based co-occurrence, still not presence-only
    vru = ((ped > 0) | (cyc > 0)).expand(PAD_US)
    events += [
        BasicEvent(name="vru_while_braking", expr=vru & hard_brake, desc="VRU visible during hard braking"),
        SequenceOfEvents(name="vru_then_braking", expressions=[vru, hard_brake], desc="VRU visible, then braking"),
    ]

for e in events:
    report.add_event(e)
print("registered events:", [e.get_name() for e in report.get_events()])

## 3. Statistics within events

Compute per-instance statistics **within the hard-braking events** — speed, deceleration,
brake pressure, and (when available) the **nearest pedestrian / vehicle distance** during each
braking event, so you see the safety context of every brake.

In [ ]:
stats_inputs = [speed, acc_x, brake, steer]
stats_names = ["vehicle_speed", "acceleration_x", "brake_pressure", "steering_angle_calculated"]
if have_distance:
    stats_inputs += [ped_dist, veh_dist]
    stats_names += ["pedestrian_nearest_distance_m", "vehicle_nearest_distance_m"]
elif have_counts:
    stats_inputs += [ped, veh]
    stats_names += ["detected_pedestrians", "detected_vehicles"]

page = Page(page_number=1)
page.add_aggregation(StatsAggregator(
    name="braking_event_stats",
    input_expressions=stats_inputs,
    channel_names=stats_names,
    statistics=["min", "mean", "max"],
    event=ev_brake,
))
report.add_page(page)
report.determine_report()
report.persist_results()

print("Event instances found per event:")
display(spark.read.table(f"{pfx}_event_instance_fact")
        .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id")
        .groupBy("event_name").count().orderBy(F.desc("count")))

print("Statistics within hard_braking events (averaged over event instances):")
display(spark.read.table(f"{pfx}_stats_aggregator_fact")
        .groupBy("channel_name", "aggregation_label")
        .agg(F.round(F.avg("statistic_value"), 3).alias("avg_over_events"))
        .orderBy("channel_name", "aggregation_label"))

## 4. Export event video clips (MP4)

For **every event name**, take its first `clips_per_event` instances, and for each instance
encode the camera frames within the event's own window
(`[start_ts, min(end_ts, start_ts+clip_max_duration_s)]` — events are `.expand()`-ed so the
window already spans a useful range) into an **H.264 MP4** on the volume. The file name
includes the event name:
`{vol_root}/<report_name>/<event_name>_<container_id>_<event_instance_id>.mp4`.

In [ ]:
import shutil
import imageio.v2 as imageio
from PIL import Image

def write_mp4(image_paths, out_path, fps=5.0, width=640):
    """Encode image files into an H.264 MP4 at out_path (must be a local, seekable path)."""
    frames = []
    for p in image_paths:
        im = Image.open(p).convert("RGB")
        w, h = im.size
        nw = width - (width % 2)                 # H.264 needs even dimensions
        nh = int(round(h * nw / w)); nh -= nh % 2
        im = im.resize((nw, max(2, nh)))
        frames.append(np.asarray(im))
    if not frames:
        raise ValueError("no frames")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    imageio.mimwrite(out_path, frames, fps=fps, codec="libx264", quality=8, macro_block_size=1)
    return out_path

def event_clip_frames(frames_pdf, start_ts, end_ts, max_duration_s=5.0, max_frames=300):
    """Frames within the event's own window [start_ts, min(end_ts, start_ts+max_duration_s)] (us)."""
    end = min(int(end_ts), int(start_ts) + int(max_duration_s * 1e6))
    sub = frames_pdf[(frames_pdf.cam_tstamp_us >= start_ts) & (frames_pdf.cam_tstamp_us <= end)]
    sub = sub.sort_values("cam_tstamp_us")
    if len(sub) > max_frames:
        idx = np.linspace(0, len(sub) - 1, max_frames).round().astype(int)
        sub = sub.iloc[idx]
    return sub

def write_event_clip(event_name, container_id, instance_id, sub_pdf, clips_dir, fps=5.0, width=640):
    """Encode frames to a local mp4 (seekable), then copy to the volume folder clips_dir.
    File name = <event_name>_<container_id>_<event_instance_id>.mp4."""
    fname = f"{event_name}_{int(container_id)}_{int(instance_id)}.mp4"
    tmp = f"/tmp/a2d2_clips/{fname}"
    write_mp4(sub_pdf.png_path.tolist(), tmp, fps=fps, width=width)
    os.makedirs(clips_dir, exist_ok=True)
    dst = f"{clips_dir}/{fname}"
    shutil.copyfile(tmp, dst)                    # finished file -> FUSE volume (no seek needed)
    return dst

In [ ]:
if not spark.catalog.tableExists(f"{pfx}_camera_frames"):
    print(f"{pfx}_camera_frames not found - re-run ingestion with download_images=true.")
else:
    frames_pdf = (spark.read.table(f"{pfx}_camera_frames")
                  .select("cam_tstamp_us", "png_path").orderBy("cam_tstamp_us").toPandas())
    ev = (spark.read.table(f"{pfx}_event_instance_fact")
          .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id"))
    counts = ev.groupBy("event_name").count().orderBy(F.desc("count")).toPandas()

    if counts.empty or frames_pdf.empty:
        print("No event instances or no camera frames available.")
    else:
        total = 0
        for ename in counts["event_name"].tolist():   # every event name
            inst = (ev.where(F.col("event_name") == ename)
                    .select("container_id", "event_instance_id", "start_ts", "end_ts")
                    .orderBy("start_ts").limit(CLIPS_PER_EVENT).toPandas())
            written = 0
            for _, r in inst.iterrows():
                sub = event_clip_frames(frames_pdf, r["start_ts"], r["end_ts"], CLIP_MAX_DURATION_S)
                if len(sub) == 0:
                    continue
                write_event_clip(ename, r["container_id"], r["event_instance_id"], sub, clips_dir, fps=CLIP_FPS)
                written += 1
            total += written
            print(f"  {ename}: wrote {written}/{min(len(inst), CLIPS_PER_EVENT)} clips")
        print(f"Wrote {total} MP4 clip(s) under {clips_dir} (named <event_name>_<container_id>_<event_instance_id>.mp4):")
        display(spark.createDataFrame(dbutils.fs.ls(clips_dir)).orderBy("name"))

## Notes
- This notebook persists **event** facts (`_event_instance_fact`, `_event_dimension`) and
  **statistics** (`_stats_aggregator_fact`/`_dimension`) — no histograms.
- Perception-fused events (`vru_while_braking`, `vru_then_braking`, …) require the
  `detected_*` channels from `a2d2_object_detection.ipynb`; without them only the bus-signal
  events run.
- Event clips are MP4 (H.264) on the volume under `{vol_root}/<report_name>/`, named
  `<event_name>_<container_id>_<event_instance_id>.mp4` — up to `clips_per_event` per event
  name. For smooth clips, ingest at a higher `images_per_second`.